# 032 — Event information and weak-label reliability / События и надёжность меток

Variant: event features=True, reliability weighting=False. Compare 031 (coordinate control), 032 (+events, identical training weights), 033 (+events, reweighted). Same training frame indices and weak D/LIVE targets as 029, including the public prefix. Corrected reference labels NEVER enter preprocessing, training, early stopping, sample selection or weighting. / Три версии: 031 координаты, 032 добавление событий при прежних весах, 033 изменение только весов. Примеры и слабые цели одинаковы, включая первые пять минут; истинные labels не используются.

Fixed models: logistic C=.01; HGB 150 trees, depth3, leaves7, leaf25, lr=.05, L2=2, early_stopping=False, seed42. Preprocessing fitted on train only: median + missing indicators; scaling only for logistic. Thresholds .5/.6/.7/.8/.9/.95/.99/.995. No retraining or threshold changes after evaluation. / Модели и пороги фиксированы заранее; после оценки не меняются.

Event features (032/033): 6 restart types and 13 phase/goal codes × active/until-start/since-end; distances capped at60s, normalized, reset per half. Five physical-context interactions: restart proximity <=15s, compatible ball location, stationary restart support, moving active-play support, moving-ball restart conflict. No absolute video clock, frame index, club identity or reference information. Event boundaries describe automatic windows, not exact whistles. Future event context is explicitly offline. / 62 признака: 57 временных по 19 типам + 5 сочетаний с геометрией и движением; часы матча и номера кадров не используются.

Weighting (033 only): weak D default .35, slow ball/players + compatible upcoming restart 1.0, moving ball + active play .1. Weak LIVE default .5, moving active play 1.0, stationary restart conflict .1. Multiply original episode weights and rebalance classes; no new labels or samples. These are hypotheses, not guaranteed truths. / Меняются только веса, без новых меток. Веса событийной надёжности — гипотеза.

Frozen H/A from 023/020 after the same D duration filter. All outputs full offline, no X. Public labels read only after all predictions are frozen; H/A was historically public-selected and ranking thresholds is public tuning, not independent validation. / H/A и фильтр неизменны, итоговые метрики диагностические.

Run All with 022 expanded_D_candidates, broadcast weak_LIVE_candidates, 008 feature cache, ATD and automatic Smart Tagging XML. Reuses 023 H/A notebook definitions. tqdm and local_review/experiments/032_event_features/progress.json track progress. Only clean code/config and trusted fitted model bundles are published; input assets, predictions and reports remain local. / Нужны локальные данные; отчёты и прогнозы остаются в local_review.


In [ ]:
EXPERIMENT='032_event_features'
MODEL_CONFIGS=[{'name': 'logistic_c001', 'kind': 'logistic', 'C': 0.01}, {'name': 'boost_d3_l7', 'kind': 'boost', 'depth': 3, 'leaves': 7, 'leaf': 25, 'trees': 150, 'l2': 2.0}]
import os
os.environ.setdefault('OMP_NUM_THREADS','4')
os.environ.setdefault('LOKY_MAX_CPU_COUNT','4')
from pathlib import Path
import json,sys,pickle,time,hashlib,warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.exceptions import ConvergenceWarning
import torch
from torch import nn
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'games.py').exists())
sys.path.insert(0,str(ROOT))
from games import GAME,half_video_blocks
from evaluation import possession_metrics as pm
OUT=ROOT/'local_review/experiments'/EXPERIMENT;OUT.mkdir(parents=True,exist_ok=True)
BASE_OUT=OUT;N=199319;FPS=25;BLOCKS=half_video_blocks(GAME);SEED=42
THRESHOLDS=[.5,.6,.7,.8,.9,.95,.99,.995]
CONTEXT='D: automatic broadcast weak training on all half-block frames INCLUDING public inputs; NO reference labels in fitting; fixed logistic C=.01 and HGB depth3/150 trees; event-feature/weight ablation 031-033; H/A historically public-selected; full offline context; transductive public diagnostics, not independent test'
def progress(stage,**kw):
 (BASE_OUT/'progress.json').write_text(json.dumps(dict(stage=stage,**kw),indent=2))
def save_json(path,value):path.write_text(json.dumps(value,ensure_ascii=False,indent=2)+'\n')
def file_hash(path):return hashlib.sha256(path.read_bytes()).hexdigest()
# Use automatic weak labels across BOTH complete half blocks, INCLUDING public frames.
# Берём автоматические слабые метки обоих таймов, ВКЛЮЧАЯ первые пять минут.
# Reference H/A/D is never used here. Unknown/conflicting automatic labels stay excluded.
# Истинные H/A/D здесь не читаются; неизвестные и конфликтные слабые метки исключены.
parent=ROOT/'local_review/experiments/022_broadcast_expanded_weak'
alignment=ROOT/'local_review/broadcast_alignment'
expanded_d=np.load(parent/'expanded_D_candidates.npy')
automatic_live=np.load(alignment/'weak_LIVE_candidates.npy')
graded=np.zeros(N,bool)
for _,first,last in BLOCKS:graded[first:last+1]=True
conflict=expanded_d&automatic_live
positive=expanded_d&~conflict&graded
negative=automatic_live&~conflict&graded
assert not (positive&negative).any()
weak_y=np.full(N,-1,np.int8);weak_y[negative]=0;weak_y[positive]=1
indices=[];weights=[];groups=[];group=0
for label in [0,1]:
 mask=weak_y==label
 starts=np.flatnonzero(mask&np.r_[True,~mask[:-1]])
 stops=np.flatnonzero(mask&np.r_[~mask[1:],True])+1
 for first,stop in zip(starts,stops):
  chosen=np.arange(first,stop,5)
  if len(chosen)>50:chosen=chosen[np.linspace(0,len(chosen)-1,50).astype(int)]
  indices.extend(chosen.tolist());weights.extend([1/len(chosen)]*len(chosen));groups.extend([group]*len(chosen));group+=1
train_indices=np.asarray(indices,dtype=int);train_y=weak_y[train_indices]
sample_weight=np.asarray(weights,dtype=float)
np.savez(BASE_OUT/'weak_training_selection.npz',indices=train_indices,labels=train_y,weights=sample_weight,group_ids=np.asarray(groups))
for label in [0,1]:sample_weight[train_y==label]/=sample_weight[train_y==label].sum()
sample_weight*=len(sample_weight)/sample_weight.sum()
assert set(np.unique(train_y))=={0,1}
assert np.array_equal(train_y,weak_y[train_indices])
in_public=(train_indices>=11185)&(train_indices<18685)
assert in_public.any(), 'Public input frames must now participate in automatic weak training'
selection_summary=dict(eligible_D_frames=int(positive.sum()),eligible_LIVE_frames=int(negative.sum()),
 training_samples=len(train_y),public_training_samples=int(in_public.sum()),
 public_weak_D_samples=int((in_public&(train_y==1)).sum()),public_weak_LIVE_samples=int((in_public&(train_y==0)).sum()),
 public_eligible_D_frames=int(positive[11185:18685].sum()),public_eligible_LIVE_frames=int(negative[11185:18685].sum()),
 excluded_conflicting_frames=int((conflict&graded).sum()),reference_labels_used_for_training=False,
 weak_D_sha256=file_hash(parent/'expanded_D_candidates.npy'),weak_LIVE_sha256=file_hash(alignment/'weak_LIVE_candidates.npy'))
save_json(BASE_OUT/'weak_label_summary.json',selection_summary)
print('Automatic-label selection / Автоматическая разметка:',selection_summary)
cache=ROOT/'local_review/experiments/008_weak_dead_boosting'
raw_features=np.load(cache/'features.npy',mmap_mode='r')
feature_names=json.loads((cache/'feature_names.json').read_text())
assert raw_features.shape==(N,92) and len(feature_names)==92
USE_EVENT_FEATURES=True
REWEIGHT=False
from starter.load_smart_tagging import load as load_smart_tagging
from games import GAMES
# Automatic tagging is provided input, NOT corrected possession reference labels.
# Автоматические события — входные данные, НЕ истинная разметка владения.
events=load_smart_tagging()
restart_types={'throw_in':'Throw-in','corner':'Corner Kick','goal_kick':'Goal Kick',
 'free_kick':'Free Kick','kickoff_goal':'Kick Off (After Goal)','kickoff_start':'Kick Off (Start)'}
phase_codes=['PLAYERS IN THE BOX','BALL IN FINAL THIRD','BALL IN THE BOX','PROGRESSION',
 'DEFENDING IN MIDDLE THIRD','CREATING CHANCES','DEFENDING IN DEFENSIVE THIRD','LONG BALL',
 'ATTACKING TRANSITION','DEFENSIVE TRANSITION','BUILD UP','DEFENDING IN ATTACKING THIRD','GOALS']
TIME_CAP_SECONDS=60.

def interval_signals(length,intervals):
 # Half-open intervals; future evidence is allowed only within the SAME half.
 # Полуоткрытые интервалы; будущее событие берём только в ТОМ ЖЕ тайме.
 active=np.zeros(length,bool);until=np.full(length,TIME_CAP_SECONDS);since=until.copy()
 if not intervals:return active,until,since
 starts=np.sort(np.array([a for a,b in intervals],int));ends=np.sort(np.array([b for a,b in intervals],int))
 changes=np.zeros(length+1,int)
 for a,b in intervals:changes[a]+=1;changes[b]-=1
 active=np.cumsum(changes[:-1])>0
 positions=np.arange(length)
 right=np.searchsorted(starts,positions,side='left');ok=right<len(starts)
 until[ok]=np.minimum((starts[right[ok]]-positions[ok])/FPS,TIME_CAP_SECONDS)
 left=np.searchsorted(ends,positions,side='right')-1;ok=left>=0
 since[ok]=np.minimum((positions[ok]-ends[left[ok]])/FPS,TIME_CAP_SECONDS)
 # During an interval, both distances are zero and 'active' disambiguates it.
 # Внутри интервала расстояния нулевые, отдельный active сохраняет смысл.
 until[active]=0.;since[active]=0.
 return active,until,since
# Synthetic checks for boundaries, no events, overlap and capped missing context.
# Искусственные проверки границ, отсутствия событий и перекрытий.
a,u,s=interval_signals(100,[(25,50),(45,60)])
assert a.sum()==35 and a[25] and not a[60] and u[0]==1 and s[60]==0
assert not interval_signals(10,[])[0].any()
assert np.all(interval_signals(10,[])[1]==60)

event_features={};signals={};event_counts={}
def add_group(name,chosen):
 active=np.zeros(N,bool);until=np.full(N,TIME_CAP_SECONDS);since=until.copy()
 for half,first,last in BLOCKS:
  half_tag='1st Half' if half==1 else '2nd Half'
  intervals=[]
  for event in chosen:
   if event['tags'].get('Half') not in [None,half_tag]:continue
   a=max(first,int(event['start_frame']));b=min(last+1,int(event['end_frame']))
   if a<b:intervals.append((a-first,b-first))
  active[first:last+1],until[first:last+1],since[first:last+1]=interval_signals(last-first+1,intervals)
 signals[name]=(active,until,since);event_counts[name]=len(chosen)
 event_features['event_'+name+'_active']=active.astype(np.float32)
 event_features['event_'+name+'_until_start_60s']=until/60
 event_features['event_'+name+'_since_end_60s']=since/60
for key,value in restart_types.items():
 add_group(key,[e for e in events if (e['code'] or '').upper()=='SET PIECES' and (e['tags'].get('Type') or '').lower()==value.lower()])
for code in phase_codes:
 add_group(code.lower().replace(' ','_'),[e for e in events if (e['code'] or '').upper()==code])
# Distances describe event windows, not exact whistle or ball-contact times.
# Времена относятся к окнам событий, а не точному свистку или касанию мяча.
col=lambda name:raw_features[:,feature_names.index(name)]
ball_speed=col('ball_speed_25');player_speed=col('median_player_speed')
slow=(col('ball_seen')==1)&(ball_speed<=.5)&(player_speed<=1.2)
moving=(col('ball_seen')==1)&(ball_speed>=2)&(ball_speed<=35)&(player_speed>=.8)
pitch=np.asarray(GAMES[GAME]['pitch'])
center_distance=np.hypot((col('ball_x')-.5)*pitch[0],(col('ball_y')-.5)*pitch[1])
location={'throw_in':np.abs(col('ball_touchline_distance'))<=3,
 'corner':col('ball_corner_distance')<=6,'goal_kick':np.abs(col('ball_goal_line_distance'))<=18,
 'free_kick':col('ball_seen')==1,'kickoff_goal':center_distance<=8,'kickoff_start':center_distance<=8}
near_restart=np.zeros(N,bool);compatible_restart=np.zeros(N,bool)
for key in restart_types:
 active,until,since=signals[key]
 near=active|(until<=15)
 near_restart|=near;compatible_restart|=near&location[key]
active_play=np.zeros(N,bool)
for key in ['progression','long_ball','attacking_transition','creating_chances','build_up']:
 active_play|=signals[key][0]
strong_event_D=compatible_restart&slow
strong_event_LIVE=active_play&moving
for name,value in [('near_restart',near_restart),('compatible_restart_location',compatible_restart),
 ('restart_stationary_support',strong_event_D),('phase_moving_support',strong_event_LIVE),
 ('restart_moving_conflict',near_restart&moving)]:event_features['event_'+name]=value.astype(np.float32)
extra=np.column_stack(list(event_features.values())).astype(np.float32)
assert np.isfinite(extra).all() and extra.shape[0]==N
if USE_EVENT_FEATURES:
 raw_features=np.column_stack([raw_features,extra]).astype(np.float32)
 feature_names=feature_names+list(event_features)
if REWEIGHT:
 # Reliability multipliers modify weights, NEVER target labels or selected frames.
 # Надёжность меняет только веса, НЕ целевые метки или состав выборки.
 confidence=np.ones(N,float)
 confidence[positive]=.35
 confidence[positive&strong_event_D]=1.
 confidence[positive&strong_event_LIVE]=.1
 confidence[negative]=.5
 confidence[negative&strong_event_LIVE]=1.
 confidence[negative&strong_event_D]=.1
 sample_weight*=confidence[train_indices]
 for label in [0,1]:sample_weight[train_y==label]/=sample_weight[train_y==label].sum()
 sample_weight*=len(sample_weight)/sample_weight.sum()
 assert np.all(sample_weight>0)
 weight_report={}
 for label in [0,1]:
  values,counts=np.unique(confidence[train_indices[train_y==label]],return_counts=True)
  weight_report[str(label)]={str(v):int(c) for v,c in zip(values,counts)}
 save_json(BASE_OUT/'reliability_summary.json',weight_report)
 print('Reliability multipliers / Множители надёжности:',weight_report)
save_json(BASE_OUT/'event_feature_summary.json',dict(event_counts=event_counts,added_feature_names=list(event_features),
 seconds_cap=60,near_restart_seconds=15,absolute_clock_features=False,team_identity_features=False,
 future_events_used=True,reference_labels_used=False))
print('Event features / Событийные признаки:',len(event_features),'types',len(event_counts))
np.save(BASE_OUT/'effective_training_weights.npy',sample_weight)
save_json(BASE_OUT/'feature_names.json',feature_names)

# Fit imputation and scaling only on training observations; expose missingness.
# Заполнение пропусков и масштабирование обучаем только на обучающих наблюдениях.
imputer=SimpleImputer(strategy='median',add_indicator=True,keep_empty_features=True)
train_imputed=imputer.fit_transform(raw_features[train_indices])
scaler=StandardScaler().fit(train_imputed)
scaled_train=scaler.transform(train_imputed).astype(np.float32)
input_width=scaled_train.shape[1]
# Freeze settings BEFORE any public evaluation. No public label CSV is read here.
# Фиксируем настройки ДО оценки; CSV публичных меток здесь не читается.
freeze=dict(experiment=EXPERIMENT,models=MODEL_CONFIGS,thresholds=THRESHOLDS,
 training_samples=len(train_y),training_D=int(train_y.sum()),training_LIVE=int((train_y==0).sum()),
 raw_features=len(feature_names),input_width=input_width,event_features=USE_EVENT_FEATURES,reweighted_weak_labels=REWEIGHT,effective_weights_sha256=file_hash(BASE_OUT/'effective_training_weights.npy'),selection_sha256=file_hash(BASE_OUT/'weak_training_selection.npz'),
 public_exclusion=[],public_inputs_used_for_weak_training=True,reference_labels_used_for_training=False,seed=SEED,context=CONTEXT,
 preprocessing='training-only median imputation + missing indicators; standardization for logistic',
 
 duration_filter=dict(max_live_gap_frames=10,min_dead_frames=25))
save_json(BASE_OUT/'frozen_configuration.json',freeze)
print('Training summary / Обучение:',{k:freeze[k] for k in ['training_samples','training_D','training_LIVE','raw_features','input_width']})


In [ ]:
torch.set_num_threads(4)
def make_network(width,hidden,dropout):
 layers=[];previous=width
 for size in hidden:
  layers.extend([nn.Linear(previous,size),nn.ReLU()])
  if dropout:layers.append(nn.Dropout(dropout))
  previous=size
 layers.append(nn.Linear(previous,1))
 return nn.Sequential(*layers)

def infer_bundle(bundle,frame_ids):
 # Chunked inference avoids storing multiple full standardized match matrices.
 # Пакетный инференс не хранит несколько полных матриц признаков матча.
 result=np.empty(len(frame_ids),np.float32)
 if bundle['kind']=='mlp':
  net=make_network(bundle['input_width'],bundle['config']['hidden'],bundle['config']['dropout'])
  net.load_state_dict({k:torch.from_numpy(v) for k,v in bundle['state'].items()});net.eval()
 for start in range(0,len(frame_ids),8192):
  stop=min(start+8192,len(frame_ids));z=bundle['imputer'].transform(raw_features[frame_ids[start:stop]])
  if bundle['kind'] not in ['forest','boost']:z=bundle['scaler'].transform(z)
  if bundle['kind']=='mlp':
   with torch.no_grad():p=net(torch.tensor(z,dtype=torch.float32)).squeeze(1).sigmoid().numpy()
  else:p=bundle['model'].predict_proba(z)[:,1]
  result[start:stop]=p
 assert np.isfinite(result).all() and ((result>=0)&(result<=1)).all()
 return result

trained=[]
for cfg in tqdm(MODEL_CONFIGS,desc=EXPERIMENT+' models'):
 np.random.seed(SEED);torch.manual_seed(SEED)
 model_dir=BASE_OUT/cfg['name'];model_dir.mkdir(exist_ok=True)
 progress('training',model=cfg['name'])
 bundle=dict(kind=cfg['kind'],config=cfg,imputer=imputer,scaler=scaler,input_width=input_width,feature_names=feature_names,event_features=USE_EVENT_FEATURES)
 if cfg['kind']=='mlp':
  net=make_network(input_width,cfg['hidden'],cfg['dropout'])
  optimizer=torch.optim.AdamW(net.parameters(),lr=.001,weight_decay=cfg['weight_decay'])
  x_tensor=torch.from_numpy(scaled_train);y_tensor=torch.tensor(train_y,dtype=torch.float32)
  w_tensor=torch.tensor(sample_weight,dtype=torch.float32)
  generator=torch.Generator().manual_seed(SEED);losses=[]
  for epoch in tqdm(range(cfg['epochs']),desc=cfg['name']+' epochs',leave=False):
   net.train();order=torch.randperm(len(train_y),generator=generator);loss_total=0.
   for batch in order.split(256):
    optimizer.zero_grad();logits=net(x_tensor[batch]).squeeze(1)
    loss=(nn.functional.binary_cross_entropy_with_logits(logits,y_tensor[batch],reduction='none')*w_tensor[batch]).mean()
    assert torch.isfinite(loss)
    loss.backward();optimizer.step();loss_total+=float(loss.detach())*len(batch)
   losses.append(loss_total/len(train_y))
   progress('training',model=cfg['name'],epoch=epoch+1,total_epochs=cfg['epochs'],loss=losses[-1])
  bundle['state']={k:v.detach().cpu().numpy() for k,v in net.state_dict().items()}
  save_json(model_dir/'training_losses.json',losses)
  print(cfg['name'],'parameters',sum(p.numel() for p in net.parameters()),'first/last loss',losses[0],losses[-1])
 elif cfg['kind']=='logistic':
  model=LogisticRegression(C=cfg['C'],max_iter=5000,solver='lbfgs',random_state=SEED)
  with warnings.catch_warnings():
   warnings.simplefilter('error',ConvergenceWarning)
   model.fit(scaled_train,train_y,sample_weight=sample_weight)
  bundle['model']=model
 elif cfg['kind']=='boost':
  model=HistGradientBoostingClassifier(max_iter=cfg['trees'],learning_rate=.05,
   max_depth=cfg['depth'],max_leaf_nodes=cfg['leaves'],min_samples_leaf=cfg['leaf'],
   l2_regularization=cfg['l2'],early_stopping=False,random_state=SEED)
  model.fit(train_imputed,train_y,sample_weight=sample_weight);bundle['model']=model
 else:
  model=RandomForestClassifier(n_estimators=cfg['trees'],max_depth=cfg['depth'],min_samples_leaf=cfg['leaf'],
    max_features='sqrt',n_jobs=4,random_state=SEED)
  model.fit(train_imputed,train_y,sample_weight=sample_weight);bundle['model']=model
 with (model_dir/'model.pkl').open('wb') as handle:pickle.dump(bundle,handle)
 progress('full inference',model=cfg['name'])
 probability=infer_bundle(bundle,np.arange(N));np.save(model_dir/'p_D.npy',probability)
 # Trusted own-model serialization check, not a test on training labels.
 # Проверка сохранения своей модели, не оценка по обучающим меткам.
 with (model_dir/'model.pkl').open('rb') as handle:restored=pickle.load(handle)
 assert np.allclose(infer_bundle(restored,np.arange(0,N,251)),probability[::251],atol=1e-6)
 trained.append(dict(config=cfg,model_sha256=file_hash(model_dir/'model.pkl'),probability_sha256=file_hash(model_dir/'p_D.npy')))
save_json(BASE_OUT/'trained_models.json',trained)


In [ ]:
# Reuse only preparation/definitions from 023, never its public evaluation cell.
# Берём только подготовку и определения 023, без его ячейки публичных метрик.
parent023=json.loads((ROOT/'sol/023_weak022_anchors020/experiment.ipynb').read_text())
ha_source=''.join(parent023['cells'][2]['source'])
prepare_source,run_source=ha_source.split('# Replace legacy D completely before running the H/A solver.',1)
prepare_source=prepare_source.replace('023_weak022_anchors020',EXPERIMENT)
run_source=run_source.split('# Freeze the complete output before opening permitted reference labels.')[0]
# Neutral D exists only during geometry preparation, before per-threshold inference.
# Нулевая D-маска используется только при подготовке геометрии.
filtered=np.zeros(N,bool);dead=filtered.copy()
exec(compile(prepare_source,'023 frozen H/A preparation','exec'))
assert OUT==BASE_OUT

def duration_filter(raw):
 joined=raw.copy()
 for a,b in runs_of(raw):
  if not raw[a] and a>0 and b<len(raw) and b-a<=10:joined[a:b]=True
 result=joined.copy()
 for a,b in runs_of(joined):
  if joined[a] and b-a<25:result[a:b]=False
 return result

manifest=[]
for cfg in MODEL_CONFIGS:
 probability=np.load(BASE_OUT/cfg['name']/'p_D.npy')
 for threshold in tqdm(THRESHOLDS,desc=cfg['name']+' thresholds + H/A'):
  threshold_tag=f't{int(round(threshold*1000)):04d}'
  variant=cfg['name']+'/'+threshold_tag
  OUT=BASE_OUT/variant;OUT.mkdir(parents=True,exist_ok=True)
  filtered=np.zeros(N,bool)
  for _,a,b in BLOCKS:filtered[a:b+1]=duration_filter(probability[a:b+1]>=threshold)
  # Runs H/A and writes exact current predictions with no extra smoothing.
  # Запускаем H/A и сохраняем точный итоговый прогноз без дополнительного сглаживания.
  exec(compile(run_source,'023 frozen H/A inference','exec'))
  assert not (labels=='X').any()
  assert np.array_equal(labels[graded]=='D',filtered[graded])
  manifest.append(dict(model=cfg['name'],threshold=threshold,variant=variant,
    prediction_sha256=file_hash(OUT/'submission.csv')))
OUT=BASE_OUT
save_json(BASE_OUT/'predictions_frozen_before_public_evaluation.json',manifest)
progress('predictions frozen',variants=len(manifest))


In [ ]:
# First public-label access in this notebook: every model and prediction is frozen.
# Первое чтение публичного эталона: все модели и прогнозы уже зафиксированы.
gt=pm.read_label_csv(ROOT/'evaluation/gt/BAR-ATH_possession_gt_public.csv')
public=gt.loc[gt.label.isin(['H','A','D']),'frame'].to_numpy()
assert len(public)==7500 and np.intersect1d(public,train_indices).size>0
# Overlap is intentional; training answers remain automatic broadcast labels.
# Пересечение намеренное; ответы обучения — только автоматические broadcast-метки.
truth=gt.set_index('frame')
def score_frames(prediction,frames):
 y=pm.label_codes(truth.loc[frames,'label'].to_numpy())
 p=pm.label_codes(prediction.loc[frames,'label'].to_numpy())
 valid_runs=pm.valid_runs(y);cm=pm.scored_confusion(pm.confusion(y,p))
 result=dict(spellF1=pm.spell_scores(y,pm.fill_abstentions(p,valid_runs),valid_runs)['mean_f1'],
  macroF1=pm.macro_f1(cm),classes={})
 for i,label in enumerate(['H','A','D']):
  precision,recall,f1=pm.per_class_prf(cm,i)
  result['classes'][label]=dict(precision=precision,recall=recall,F1=f1,
    support=int(cm[i].sum()),predicted=int(cm[:,i].sum()))
 assert np.isclose(np.mean([r['F1'] for r in result['classes'].values()]),result['macroF1'])
 return result

comparison=[];registry=[]
for entry in manifest:
 variant_dir=BASE_OUT/entry['variant']
 assert file_hash(variant_dir/'submission.csv')==entry['prediction_sha256']
 prediction=pm.read_label_csv(variant_dir/'submission.csv').set_index('frame')
 metrics=dict(context=CONTEXT)
 for window,frames in [('public_5min',public),('last_minute',public[6000:])]:
  metric=score_frames(prediction,frames);metrics[window]=metric;d=metric['classes']['D']
  values=prediction.loc[frames,'label'].to_numpy()
  row=dict(model=entry['model'],threshold=entry['threshold'],window=window,
    spellF1=metric['spellF1'],macroF1=metric['macroF1'],
    D_precision=d['precision'],D_recall=d['recall'],D_F1=d['F1'])
  for label in ['H','A','D','X']:
   row[label+'_runs']=int(sum(values[a]==label for a,b in runs_of(values)))
   row[label+'_frames']=int((values==label).sum())
  comparison.append(row)
 save_json(variant_dir/'metrics.json',metrics)
 registry.append([EXPERIMENT[:3]+'-'+entry['variant'].replace('/','-'),
   entry['model']+' + H/A, t='+str(entry['threshold']),
   EXPERIMENT+'/'+entry['variant']+'/submission.csv',None,CONTEXT])
save_json(BASE_OUT/'experiment_registry.json',registry)
pd.DataFrame(comparison).to_csv(BASE_OUT/'comparison.csv',index=False)
save_json(BASE_OUT/'comparison.json',comparison)
# Ranking is descriptive public tuning, not a claim of independent performance.
# Ранжирование — описательная подстройка по публичному фрагменту, не независимая оценка.
public_rows=pd.DataFrame(comparison).query("window == 'public_5min'")
best_rows=public_rows.sort_values(['spellF1','macroF1'],ascending=False).groupby('model',sort=False).head(1)
lines=['# '+EXPERIMENT+' — все варианты','',CONTEXT,'',
 'Все архитектуры и пороги зафиксированы заранее. Обучение на полной автоматической разметке, включая первые пять минут, без истинных labels. Сравнение 031–033 отделяет эффект событийных признаков и весов. Сравнение порогов на публичном фрагменте — подбор, не независимая проверка. H/A исторически подбирался по публичным меткам.',
 '', '| Модель | Порог | spellF1 | macroF1 | D precision | D recall | D F1 | D-серий |','|---|---:|---:|---:|---:|---:|---:|---:|']
for r in public_rows.to_dict('records'):
 lines.append(f"| {r['model']} | {r['threshold']} | {r['spellF1']:.4f} | {r['macroF1']:.4f} | {r['D_precision']:.4f} | {r['D_recall']:.4f} | {r['D_F1']:.4f} | {r['D_runs']} |")
(BASE_OUT/'REVIEW_RU.md').write_text('\n'.join(lines)+'\n')
progress('completed',variants=len(manifest),model_count=len(MODEL_CONFIGS))
print('Best spellF1 row per model; public-tuned diagnostics / Лучшие по spellF1, подстроенная диагностика:')
print(best_rows.to_string(index=False))
